In [85]:
# 에콰도르 소매 데이터 전처리: 단일 df.parquet 생성
import pandas as pd
import numpy as np
from pathlib import Path

# 데이터 경로 (C:\Users\kjh\data)
DATA_DIR = Path(r"C:\Users\kjh\data")
OUTPUT_PATH = Path(r"C:\Users\kjh\df.parquet")

pd.set_option("display.max_columns", None)

In [86]:
# 1) train.csv 로드 (메인 시계열: date, store_nbr, family)
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
train["date"] = pd.to_datetime(train["date"])
print("train shape:", train.shape)
print(train.head())

train shape: (3000888, 6)
   id       date  store_nbr      family  sales  onpromotion
0   0 2013-01-01          1  AUTOMOTIVE    0.0            0
1   1 2013-01-01          1   BABY CARE    0.0            0
2   2 2013-01-01          1      BEAUTY    0.0            0
3   3 2013-01-01          1   BEVERAGES    0.0            0
4   4 2013-01-01          1       BOOKS    0.0            0


In [87]:
# 2) oil.csv 로드 (날짜별 유가)
oil = pd.read_csv(DATA_DIR / "oil.csv", parse_dates=["date"])
oil["date"] = pd.to_datetime(oil["date"])
# 결측 보간 (영업일만 있어 비영업일 결측 발생)
oil = oil.sort_values("date").set_index("date").resample("D").asfreq().reset_index()
oil["dcoilwtico"] = oil["dcoilwtico"].interpolate(method="linear")
print("oil shape:", oil.shape)
print(oil.head(10))

oil shape: (1704, 2)
        date  dcoilwtico
0 2013-01-01         NaN
1 2013-01-02   93.140000
2 2013-01-03   92.970000
3 2013-01-04   93.120000
4 2013-01-05   93.146667
5 2013-01-06   93.173333
6 2013-01-07   93.200000
7 2013-01-08   93.210000
8 2013-01-09   93.080000
9 2013-01-10   93.810000


In [88]:
# 3) holidays_events.csv 로드 → 날짜별 이벤트 정보 집계
holidays = pd.read_csv(DATA_DIR / "holidays_events.csv", parse_dates=["date"])
holidays["date"] = pd.to_datetime(holidays["date"])
# 날짜별 휴일/이벤트 요약 (같은 날 여러 행 → 하나로)
holidays_agg = (
    holidays.groupby("date")
    .agg(
        is_holiday=("type", lambda x: (x == "Holiday").any()),
        is_transfer=("transferred", "any"),
        event_types=("type", lambda x: "|".join(sorted(x.unique().astype(str)))),
        descriptions=("description", lambda x: "|".join(x.astype(str).unique()[:3])),
    )
    .reset_index()
)
# 휴일이 없는 날도 포함하려면 train의 date 범위로 확장 (merge 시 left join으로 처리)
print("holidays_agg shape:", holidays_agg.shape)
print(holidays_agg.head(10))

holidays_agg shape: (312, 5)
        date  is_holiday  is_transfer event_types  \
0 2012-03-02        True        False     Holiday   
1 2012-04-01        True        False     Holiday   
2 2012-04-12        True        False     Holiday   
3 2012-04-14        True        False     Holiday   
4 2012-04-21        True        False     Holiday   
5 2012-05-12        True        False     Holiday   
6 2012-06-23        True        False     Holiday   
7 2012-06-25        True        False     Holiday   
8 2012-07-03        True        False     Holiday   
9 2012-07-23        True        False     Holiday   

                                        descriptions  
0                                 Fundacion de Manta  
1                      Provincializacion de Cotopaxi  
2                                Fundacion de Cuenca  
3                          Cantonizacion de Libertad  
4                          Cantonizacion de Riobamba  
5                             Cantonizacion del Puyo  
6 

In [89]:
# 4) stores.csv 로드 (매장 마스터)
stores = pd.read_csv(DATA_DIR / "stores.csv")
# 컬럼명 일부 변경하여 merge 후 구분 가능하게 (선택)
print("stores shape:", stores.shape)
print(stores.head())

stores shape: (54, 5)
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4


In [90]:
# 5) transactions.csv 로드 (날짜·매장별 거래 건수)
transactions = pd.read_csv(DATA_DIR / "transactions.csv", parse_dates=["date"])
transactions["date"] = pd.to_datetime(transactions["date"])
print("transactions shape:", transactions.shape)
print(transactions.head())

transactions shape: (83488, 3)
        date  store_nbr  transactions
0 2013-01-01         25           770
1 2013-01-02          1          2111
2 2013-01-02          2          2358
3 2013-01-02          3          3487
4 2013-01-02          4          1922


In [91]:
# 6) 메인 df = train 기준, 시계열 키: date, store_nbr, family
df = train.copy()
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
print("df (train 기준) shape:", df.shape)

df (train 기준) shape: (3000888, 9)


In [92]:
# 7) oil 정보 merge (날짜 기준)
df = df.merge(oil[["date", "dcoilwtico"]], on="date", how="left")
print("oil merge 후 shape:", df.shape)
print(df[["date", "dcoilwtico"]].head())

oil merge 후 shape: (3000888, 10)
        date  dcoilwtico
0 2013-01-01         NaN
1 2013-01-01         NaN
2 2013-01-01         NaN
3 2013-01-01         NaN
4 2013-01-01         NaN


In [93]:
# 8) 이벤트/휴일 정보 merge (날짜 기준)
df = df.merge(holidays_agg, on="date", how="left")
df["is_holiday"] = df["is_holiday"].fillna(False)
df["is_transfer"] = df["is_transfer"].fillna(False)
print("holidays merge 후 shape:", df.shape)
print(df[["date", "is_holiday", "event_types"]].head())

holidays merge 후 shape: (3000888, 14)
        date  is_holiday event_types
0 2013-01-01        True     Holiday
1 2013-01-01        True     Holiday
2 2013-01-01        True     Holiday
3 2013-01-01        True     Holiday
4 2013-01-01        True     Holiday


C:\Users\kjh\AppData\Local\Temp\ipykernel_35904\2770376497.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_holiday"] = df["is_holiday"].fillna(False)
C:\Users\kjh\AppData\Local\Temp\ipykernel_35904\2770376497.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_transfer"] = df["is_transfer"].fillna(False)


In [94]:
# 9) 매장 정보 merge (store_nbr 기준)
df = df.merge(stores, on="store_nbr", how="left")
print("stores merge 후 shape:", df.shape)
print(df[["store_nbr", "city", "state", "type", "cluster"]].head())

stores merge 후 shape: (3000888, 18)
   store_nbr   city      state type  cluster
0          1  Quito  Pichincha    D       13
1          1  Quito  Pichincha    D       13
2          1  Quito  Pichincha    D       13
3          1  Quito  Pichincha    D       13
4          1  Quito  Pichincha    D       13


In [95]:
# 10) 거래 건수(transactions) merge (날짜·매장 기준)
df = df.merge(transactions[["date", "store_nbr", "transactions"]], on=["date", "store_nbr"], how="left")
print("transactions merge 후 shape:", df.shape)
print(df[["date", "store_nbr", "transactions"]].head())

transactions merge 후 shape: (3000888, 19)
        date  store_nbr  transactions
0 2013-01-01          1           NaN
1 2013-01-01          1           NaN
2 2013-01-01          1           NaN
3 2013-01-01          1           NaN
4 2013-01-01          1           NaN


In [96]:
# 11) 연도·날짜 기준 정렬 후 컬럼 순서 정리
df = df.sort_values(["year", "date", "store_nbr", "family"]).reset_index(drop=True)
# 시계열·식별자 → 보조 데이터 순서로 정리
cols_order = [
    "id", "date", "year", "month", "weekofyear",
    "store_nbr", "city", "state", "type", "cluster",
    "family", "sales", "onpromotion", "transactions",
    "dcoilwtico", "is_holiday", "is_transfer", "event_types", "descriptions"
]
df = df[[c for c in cols_order if c in df.columns]]
print("최종 df shape:", df.shape)
print(df.head())

최종 df shape: (3000888, 19)
   id       date  year  month  weekofyear  store_nbr   city      state type  \
0   0 2013-01-01  2013      1           1          1  Quito  Pichincha    D   
1   1 2013-01-01  2013      1           1          1  Quito  Pichincha    D   
2   2 2013-01-01  2013      1           1          1  Quito  Pichincha    D   
3   3 2013-01-01  2013      1           1          1  Quito  Pichincha    D   
4   4 2013-01-01  2013      1           1          1  Quito  Pichincha    D   

   cluster      family  sales  onpromotion  transactions  dcoilwtico  \
0       13  AUTOMOTIVE    0.0            0           NaN         NaN   
1       13   BABY CARE    0.0            0           NaN         NaN   
2       13      BEAUTY    0.0            0           NaN         NaN   
3       13   BEVERAGES    0.0            0           NaN         NaN   
4       13       BOOKS    0.0            0           NaN         NaN   

   is_holiday  is_transfer event_types        descriptions  
0   

In [97]:
# 12) 단일 parquet 파일로 저장
df.to_parquet(OUTPUT_PATH, index=False)
print(f"저장 완료: {OUTPUT_PATH}")
print("컬럼:", list(df.columns))

저장 완료: C:\Users\kjh\df.parquet
컬럼: ['id', 'date', 'year', 'month', 'weekofyear', 'store_nbr', 'city', 'state', 'type', 'cluster', 'family', 'sales', 'onpromotion', 'transactions', 'dcoilwtico', 'is_holiday', 'is_transfer', 'event_types', 'descriptions']


In [98]:
# === 고유값 및 계층 구조 확인 ===

# 1) 날짜 범위 → 월이 1~8월만 있는 이유: train 데이터가 2013-01-01 ~ 2017-08-15 까지만 있음
print("=== 날짜 범위 ===")
print("date min:", df["date"].min(), "| date max:", df["date"].max())
print("year 고유:", sorted(df["year"].unique()))
print("month 고유:", sorted(df["month"].unique()))
print("→ 2013~2017년이고, 2017년은 8월(8/15)까지만 있어서 month가 1~8만 존재\n")

# 2) 상품: 이 데이터는 SKU가 아니라 상품군(family) 단위
print("=== 상품군(family) — SKU가 아님, 카테고리 단위 ===")
families = df["family"].unique()
print(f"상품군 개수: {len(families)}개")
print("상품군 목록:")
for i, f in enumerate(sorted(families), 1):
    print(f"  {i:2d}. {f}")
print()

# 3) 매장 계층: store_nbr → city, state, type, cluster
print("=== 매장 계층 (store_nbr 기준) ===")
print(f"매장 수(store_nbr): {df['store_nbr'].nunique()}개")
print("city:", df["city"].nunique(), "개 →", sorted(df["city"].unique()))
print("state:", df["state"].nunique(), "개")
print("type (매장 유형):", df["type"].unique().tolist())
print("cluster:", df["cluster"].nunique(), "개 (1~16 등)\n")

# 4) 행 단위 요약: 한 행 = (날짜, 매장, 상품군) 1조합
print("=== 행 단위 의미 ===")
print("한 행 = 특정 날짜(date) + 특정 매장(store_nbr) + 특정 상품군(family) 의 일별 판매(sales)")
print("따라서 날짜×매장×상품군 조합 수:", df.groupby(["date", "store_nbr", "family"]).ngroups)
print("실제 행 수:", len(df))

=== 날짜 범위 ===
date min: 2013-01-01 00:00:00 | date max: 2017-08-15 00:00:00
year 고유: [np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017)]
month 고유: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
→ 2013~2017년이고, 2017년은 8월(8/15)까지만 있어서 month가 1~8만 존재

=== 상품군(family) — SKU가 아님, 카테고리 단위 ===
상품군 개수: 33개
상품군 목록:
   1. AUTOMOTIVE
   2. BABY CARE
   3. BEAUTY
   4. BEVERAGES
   5. BOOKS
   6. BREAD/BAKERY
   7. CELEBRATION
   8. CLEANING
   9. DAIRY
  10. DELI
  11. EGGS
  12. FROZEN FOODS
  13. GROCERY I
  14. GROCERY II
  15. HARDWARE
  16. HOME AND KITCHEN I
  17. HOME AND KITCHEN II
  18. HOME APPLIANCES
  19. HOME CARE
  20. LADIESWEAR
  21. LAWN AND GARDEN
  22. LINGERIE
  23. LIQUOR,WINE,BEER
  24. MAGAZINES
  25. MEATS
  26. PERSONAL CARE
  27. PET SUPPLIES
  28. PLAYERS AND ELECTRONICS
  29. POULTRY
  30. PREPARED FOODS
  31. PRODUCE
  32. SCHOO

In [99]:
# === 분석 단위 정의 및 데이터 존재 여부(커버리지) 진단 ===
# 분석 단위: 한 행 = (date, store_nbr, family) = "특정 날짜 · 특정 매장 · 특정 상품군" 의 일별 판매량
# 분석을 위해선 모든 날짜×매장×상품군 조합이 있거나, 최소한 "데이터가 있는" 조합만 남겨야 함

# 전체 기간 내 날짜 수 (이론상 각 (store_nbr, family)마다 이만큼 행이 있어야 완전함)
all_dates = df["date"].unique()
n_dates = len(all_dates)
print("분석 단위: (date, store_nbr, family) = 일별·매장별·상품군별 1행")
print(f"기간 내 전체 날짜 수: {n_dates}일 (date min ~ max)\n")

# (store_nbr, family) 조합별 실제 존재하는 날짜 수 → 커버리지
coverage = (
    df.groupby(["store_nbr", "family"])
    .agg(n_dates_actual=("date", "nunique"))
    .reset_index()
)
coverage["n_dates_expected"] = n_dates
coverage["coverage_ratio"] = coverage["n_dates_actual"] / n_dates

# 커버리지 100%인 조합만 "완전한 시계열"로 간주 (필요시 임계값 조정, 예: 0.99)
MIN_COVERAGE = 1.0  # 1.0 = 모든 날짜에 데이터 있는 조합만
valid_combos = coverage[coverage["coverage_ratio"] >= MIN_COVERAGE]
invalid_combos = coverage[coverage["coverage_ratio"] < MIN_COVERAGE]

print("=== (store_nbr, family) 조합별 날짜 커버리지 ===")
print(f"전체 조합 수: {len(coverage)} (54매장 × 33상품군 = 1782)")
print(f"커버리지 {MIN_COVERAGE*100:.0f}% 이상인 조합(분석 대상): {len(valid_combos)}개")
print(f"그 미만인 조합(제외 대상): {len(invalid_combos)}개")

if len(invalid_combos) > 0:
    print("\n[제외될 조합 예시] coverage_ratio < 1 인 (store_nbr, family):")
    print(invalid_combos.sort_values("coverage_ratio").head(10))

# 상품군(family) 단위: 해당 상품군이 "모든 매장×날짜"에서 얼마나 채워져 있는지
family_coverage = (
    df.groupby("family")
    .agg(n_rows=("date", "count"), n_dates=("date", "nunique"))
    .reset_index()
)
family_coverage["n_stores"] = df.groupby("family")["store_nbr"].nunique().values
family_coverage["expected_rows"] = n_dates * 54  # 54개 매장
family_coverage["coverage_ratio"] = family_coverage["n_rows"] / family_coverage["expected_rows"]
print("\n=== 상품군(family)별 행 수·커버리지 ===")
print(family_coverage.sort_values("coverage_ratio").to_string())

분석 단위: (date, store_nbr, family) = 일별·매장별·상품군별 1행
기간 내 전체 날짜 수: 1684일 (date min ~ max)

=== (store_nbr, family) 조합별 날짜 커버리지 ===
전체 조합 수: 1782 (54매장 × 33상품군 = 1782)
커버리지 100% 이상인 조합(분석 대상): 1782개
그 미만인 조합(제외 대상): 0개

=== 상품군(family)별 행 수·커버리지 ===
                        family  n_rows  n_dates  n_stores  expected_rows  coverage_ratio
0                   AUTOMOTIVE   90936     1684        54          90936             1.0
1                    BABY CARE   90936     1684        54          90936             1.0
2                       BEAUTY   90936     1684        54          90936             1.0
3                    BEVERAGES   90936     1684        54          90936             1.0
4                        BOOKS   90936     1684        54          90936             1.0
5                 BREAD/BAKERY   90936     1684        54          90936             1.0
6                  CELEBRATION   90936     1684        54          90936             1.0
7                     CLEANING   90936    

In [100]:
# === "데이터가 있는" (store_nbr, family)만 남기기 → 분석용 df.parquet 저장 ===
# 커버리지 MIN_COVERAGE 이상인 조합만 유지
df_analysis = df.merge(
    valid_combos[["store_nbr", "family"]],
    on=["store_nbr", "family"],
    how="inner"
)
df_analysis = df_analysis.sort_values(["year", "date", "store_nbr", "family"]).reset_index(drop=True)

print(f"필터 전 행 수: {len(df)}")
print(f"필터 후 행 수(커버리지 {MIN_COVERAGE*100:.0f}% 이상만): {len(df_analysis)}")
print(f"제거된 행: {len(df) - len(df_analysis)}")
print(f"유지된 (store_nbr, family) 조합 수: {df_analysis.groupby(['store_nbr','family']).ngroups}")

# 분석용으로 df를 이걸로 덮어쓰고 parquet 저장 (동일 경로에 저장해 최종 결과물로 사용)
df = df_analysis
df.to_parquet(OUTPUT_PATH, index=False)
print(f"\n저장 완료: {OUTPUT_PATH} (분석 가능한 조합만 포함)")

필터 전 행 수: 3000888
필터 후 행 수(커버리지 100% 이상만): 3000888
제거된 행: 0
유지된 (store_nbr, family) 조합 수: 1782

저장 완료: C:\Users\kjh\df.parquet (분석 가능한 조합만 포함)


In [101]:
# === 각 항목(컬럼)별 고유값 확인 후 Excel 추출 ===
import re

# 고유값 Excel 저장 경로
UNIQUE_EXCEL_PATH = Path(r"C:\Users\kjh\data\df_고유값_요약.xlsx")

def excel_sheet_name(name, used=None):
    """Excel 시트명: 31자 제한, 사용 불가 문자 제거"""
    used = used or set()
    s = re.sub(r'[\/*?:\[\]\\]', '_', str(name))[:31]
    base, i = s, 0
    while s in used:
        i += 1
        s = f"{base[:28]}_{i}"[:31]
    used.add(s)
    return s or "Sheet"

summary_rows = []
used_sheets = set()
with pd.ExcelWriter(UNIQUE_EXCEL_PATH, engine="openpyxl") as writer:
    # 첫 시트: 전체 요약 (컬럼명, 고유값 개수)
    for col in df.columns:
        summary_rows.append({"컬럼": col, "고유값_개수": df[col].nunique()})
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="요약_컬럼별고유값수", index=False)

    for col in df.columns:
        ser = df[col].dropna()
        n_unique = ser.nunique()
        sheet_name = excel_sheet_name(col, used_sheets)

        # 고유값이 너무 많으면 샘플만 (날짜/ID 등)
        if n_unique > 500000:
            uniques = ser.unique()
            out = pd.DataFrame({"고유값_샘플": uniques[:500000]})
            out.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            # 고유값 + 출현 횟수
            vc = df[col].value_counts(dropna=False)
            out = pd.DataFrame({"고유값": vc.index, "건수": vc.values})
            out.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"저장 완료: {UNIQUE_EXCEL_PATH}")
print("시트 구성: '요약_컬럼별고유값수' + 컬럼별 시트(고유값·건수 또는 고유값_샘플)")

저장 완료: C:\Users\kjh\data\df_고유값_요약.xlsx
시트 구성: '요약_컬럼별고유값수' + 컬럼별 시트(고유값·건수 또는 고유값_샘플)


In [102]:
# === 매장×상품군(store_nbr, family) 단위 SBC 4분류 → df2 (raw) ===
# SBC: ADI(Average Demand Interval), CV² 기준 4유형 (Smooth, Erratic, Intermittent, Lumpy)

# (store_nbr, family)별 일별 판매 시계열에서 ADI, CV² 계산
sbc_raw = (
    df.groupby(["store_nbr", "family"])
    .agg(
        n_days=("sales", "count"),
        n_positive=("sales", lambda x: (x > 0).sum()),
        mean_sales=("sales", "mean"),
        std_sales=("sales", "std"),
    )
    .reset_index()
)
# 양수 판매만으로 CV² (수요 크기 변동성)
pos_only = df[df["sales"] > 0].groupby(["store_nbr", "family"])["sales"].agg(mean_pos="mean", std_pos="std").reset_index()
sbc_raw = sbc_raw.merge(pos_only, on=["store_nbr", "family"], how="left")
sbc_raw["ADI"] = np.where(
    sbc_raw["n_positive"] > 0,
    sbc_raw["n_days"] / sbc_raw["n_positive"],
    np.nan,
)
sbc_raw["CV2"] = np.where(
    sbc_raw["mean_pos"].notna() & (sbc_raw["mean_pos"] > 0),
    (sbc_raw["std_pos"] / sbc_raw["mean_pos"]) ** 2,
    np.nan,
)

# SBC 4분류: ADI 1.32, CV² 0.49 기준 (Syntetos–Boylan 스타일)
ADI_TH, CV2_TH = 1.32, 0.49
def sbc_cluster(row):
    adi, cv2 = row["ADI"], row["CV2"]
    if pd.isna(adi) or pd.isna(cv2):
        return 4  # 결측은 Lumpy로
    if adi < ADI_TH and cv2 < CV2_TH:
        return 1  # Smooth
    if adi < ADI_TH and cv2 >= CV2_TH:
        return 2  # Erratic
    if adi >= ADI_TH and cv2 < CV2_TH:
        return 3  # Intermittent
    return 4  # Lumpy

sbc_raw["SBC_CLUSTER"] = sbc_raw.apply(sbc_cluster, axis=1)
labels = {1: "Smooth", 2: "Erratic", 3: "Intermittent", 4: "Lumpy"}
sbc_raw["SBC_CLUSTER_LABEL"] = sbc_raw["SBC_CLUSTER"].map(labels)

# df2: 매장·상품군별 SBC 결과 (raw)
df2 = sbc_raw[["store_nbr", "family", "ADI", "CV2", "SBC_CLUSTER", "SBC_CLUSTER_LABEL"]].copy()
df2 = df2.sort_values(["store_nbr", "family"]).reset_index(drop=True)
print("매장×상품군 단위 SBC 4분류 (df2):")
print(df2)
print("\nSBC_CLUSTER 분포:")
print(df2["SBC_CLUSTER_LABEL"].value_counts().sort_index())

매장×상품군 단위 SBC 4분류 (df2):
      store_nbr                      family        ADI       CV2  SBC_CLUSTER  \
0             1                  AUTOMOTIVE   1.151059  0.494641            2   
1             1                   BABY CARE        NaN       NaN            4   
2             1                      BEAUTY   1.177622  0.376967            1   
3             1                   BEVERAGES   1.003576  0.205269            1   
4             1                       BOOKS  13.259843  0.461304            3   
...         ...                         ...        ...       ...          ...   
1777         54                     POULTRY   1.003576  0.248088            1   
1778         54              PREPARED FOODS   1.002978  0.213260            1   
1779         54                     PRODUCE   1.657480  0.090016            3   
1780         54  SCHOOL AND OFFICE SUPPLIES  10.395062  1.397616            4   
1781         54                     SEAFOOD   1.652601  0.620745            4   

  

In [103]:
print(df)

              id       date  year  month  weekofyear  store_nbr       city  \
0              0 2013-01-01  2013      1           1          1      Quito   
1              1 2013-01-01  2013      1           1          1      Quito   
2              2 2013-01-01  2013      1           1          1      Quito   
3              3 2013-01-01  2013      1           1          1      Quito   
4              4 2013-01-01  2013      1           1          1      Quito   
...          ...        ...   ...    ...         ...        ...        ...   
3000883  3000751 2017-08-15  2017      8          33         54  El Carmen   
3000884  3000752 2017-08-15  2017      8          33         54  El Carmen   
3000885  3000753 2017-08-15  2017      8          33         54  El Carmen   
3000886  3000754 2017-08-15  2017      8          33         54  El Carmen   
3000887  3000755 2017-08-15  2017      8          33         54  El Carmen   

             state type  cluster                      family   